<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/Logistic_Regression_5class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#All imports
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


In [ ]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [ ]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [ ]:

#All the price features which include values as medium and medium_low
valuation_features = [
    'Shillers Cyclically Adjusted P/E Ratio',
    'Enterprise Value Multiple',
    'Price/Book',
    'Price/Sales',
    'Price/Cash flow',
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',
    'Trailing P/E to Growth (PEG) ratio',
    'Book/Market',
    'Dividend Yield',
    'Dividend Payout Ratio'
]

# Select _Regime columns
regime_cols = [f"{col}_Regime" for col in valuation_features if f"{col}_Regime" in train.columns]

# Convert to string
for col in regime_cols:
    train[col] = train[col].astype(str)
    val[col] = val[col].astype(str)

# Missing values
for col in regime_cols:
    train[col] = train[col].replace("nan", "Missing")
    val[col] = val[col].replace("nan", "Missing")

# One-hot encoding (
train = pd.get_dummies(train, columns=regime_cols, drop_first=True)
val = pd.get_dummies(val, columns=regime_cols, drop_first=True)

# Align val and test on train columns
val = val.reindex(columns=train.columns, fill_value=0)


In [ ]:
#making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After"])
y_train = train["Return_class"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After"])
y_val = val["Return_class"]



In [ ]:
#dropping columns
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])


In [ ]:

#Making imputer
imputer = SimpleImputer(strategy='mean')

# Fit on training
X_train_scaled = imputer.fit_transform(X_train)

# Transform val
X_val_scaled = imputer.transform(X_val)


In [ ]:
# stratified K-fold for logistic regression
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
report_list = []

fold = 1
for train_index, val_index in skf.split(X_train_scaled, y_train):
    X_tr, X_val_fold = X_train_scaled[train_index], X_train_scaled[val_index]
    y_tr, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val_fold)

    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average='macro')
    report = classification_report(y_val_fold, y_pred, output_dict=True)

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)
    report_list.append(report)

    print(f"Fold {fold} - Accuracy: {acc:.4f}, Macro F1: {macro_f1:.4f}")
    fold += 1

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 1 - Accuracy: 0.7691, Macro F1: 0.1822


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 2 - Accuracy: 0.7685, Macro F1: 0.1775


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 3 - Accuracy: 0.7652, Macro F1: 0.1736


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold 4 - Accuracy: 0.7673, Macro F1: 0.1737
Fold 5 - Accuracy: 0.7657, Macro F1: 0.1812


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# average score
print("\nAverage Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# average classification report per class
avg_report = {}
classes = y_train.unique()
for cls in classes:
    avg_report[str(cls)] = {
        'precision': np.mean([r[str(cls)]['precision'] for r in report_list]),
        'recall': np.mean([r[str(cls)]['recall'] for r in report_list]),
        'f1-score': np.mean([r[str(cls)]['f1-score'] for r in report_list]),
        'support': np.mean([r[str(cls)]['support'] for r in report_list])
    }

avg_report_df = pd.DataFrame(avg_report).T
print("\nAverage Classification Report per class:")
print(avg_report_df)


Average Accuracy: 0.7671686560342104
Average Macro F1: 0.17764844958043913

Average Classification Report per class:
      precision    recall  f1-score  support
0.0    0.772347  0.992400  0.868653   1394.8
1.0    0.000000  0.000000  0.000000    120.0
-1.0   0.122222  0.003738  0.007118    106.8
-2.0   0.100000  0.004444  0.008511     89.6
2.0    0.028571  0.002128  0.003960     94.4
